# The Heritage Short in Google Colab

This notebook uses Colab as a hosted Linux machine. For the current film, the repository already contains the approved images, clips, narration, and score, so `make film` rebuilds both exports without API keys or a GPU.

To make a new story, add Gemini and Cloudflare credentials through Colab Secrets, supply an instrumental score, then run the optional regeneration cell. The video stage here is FFmpeg pan-and-zoom, the documented fallback when Kaggle/Wan is unavailable.

In [ ]:
%cd /content
!if [ ! -d heritage-short/.git ]; then git clone https://github.com/abeni505/AI-Content-Generator-The-Heritage-Short.git heritage-short; else git -C heritage-short pull --ff-only; fi
%cd /content/heritage-short

In [ ]:
!apt-get update -qq && apt-get install -y -qq ffmpeg fonts-liberation make
!python -m pip install -q -r requirements.txt
!ffmpeg -hide_banner -filters 2>/dev/null | grep -E 'loudnorm|sidechaincompress'

In [ ]:
!make film

In [ ]:
from google.colab import files
files.download('outputs/final/original_score/heritage_short_9x16.mp4')
files.download('outputs/final/original_score/heritage_short_16x9.mp4')

## Optional: regenerate missing or changed assets

Create Colab Secrets named `GEMINI_API_KEY`, `CLOUDFLARE_ACCOUNT_ID`, and `CLOUDFLARE_API_TOKEN` before running this section. The code writes them only to the temporary Colab runtime's ignored `.env` file. Regeneration may incur provider usage. To force all existing keyframes and clips to be regenerated, change the last command to `!make regenerate FORCE=1`. For a single shot, use `!make shot SHOT=shot_03`. A new project should supply its own score file and set `MUSIC` to that path.

In [ ]:
from google.colab import userdata
from pathlib import Path
import shlex
import os

secret_names = ['GEMINI_API_KEY', 'CLOUDFLARE_ACCOUNT_ID', 'CLOUDFLARE_API_TOKEN']
secrets = {name: userdata.get(name) for name in secret_names}
missing = [name for name, value in secrets.items() if not value]
if missing:
    raise RuntimeError('Add these values to Colab Secrets first: ' + ', '.join(missing))
Path('.env').write_text(''.join(f'{name}={shlex.quote(value)}\n' for name, value in secrets.items()))
os.chmod('.env', 0o600)
print('Credentials loaded into this temporary runtime.')

In [ ]:
!make regenerate